# Phase 19: Feedback Logger

**File:** `19_feedback_logger.py`

**Purpose:** Store thumbs-up/down feedback from users.

This notebook imports the reusable Phase 19 logger, previews the Phase 15
Streamlit transcript, writes privacy-conscious feedback artifacts, validates the
generated `19_` files, and displays feedback plots.

## 1. Load Shared Logger Module

The resolver works from the workspace root, project root, or `13_notebooks` folder.

In [1]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path


def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "01_data").is_dir() and (candidate / "09_evaluation").is_dir():
            return candidate
        nested = candidate / "hospital_patient_helpdesk_chatbot"
        if (nested / "01_data").is_dir() and (nested / "09_evaluation").is_dir():
            return nested
    raise FileNotFoundError("Could not locate hospital_patient_helpdesk_chatbot project root.")


PROJECT_ROOT = resolve_project_root()
MODULE_PATH = PROJECT_ROOT / "09_evaluation" / "19_feedback_logger.py"
TRANSCRIPT_PATH = PROJECT_ROOT / "01_data" / "processed" / "15_streamlit_transcript.json"

spec = importlib.util.spec_from_file_location("feedback_logger_phase19", MODULE_PATH)
feedback_logger = importlib.util.module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = feedback_logger
spec.loader.exec_module(feedback_logger)

print(f"Project root: {PROJECT_ROOT}")
print(f"Transcript: {TRANSCRIPT_PATH}")
print(f"Module: {MODULE_PATH}")

Project root: C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot
Transcript: C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\15_streamlit_transcript.json
Module: C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\09_evaluation\19_feedback_logger.py


## 2. Inspect Transcript Input

The transcript provides safe demo turns from the Streamlit UI evaluation.

In [2]:
transcript = feedback_logger.read_transcript(TRANSCRIPT_PATH)
print(f"Transcript turns: {len(transcript)}")
for turn in transcript:
    response = turn.get("response", {})
    print(turn["turn_id"], response.get("mode"), response.get("guardrail_action"), "-", turn["question"])

Transcript turns: 3
TURN-001 grounded_answer pass - How can I book an appointment?
TURN-002 grounded_answer pass - Where is the cardiology department?
TURN-003 emergency override - I have severe chest pain. What is wrong with me?


## 3. Generate Feedback Artifacts

The notebook calls `log_feedback()` from the Python module.

In [3]:
config = feedback_logger.FeedbackLoggerConfig.from_project_root(PROJECT_ROOT)
result = feedback_logger.log_feedback(config)

print(f"Total feedback records: {result.total_feedback}")
print(f"Failed feedback records: {result.failed_feedback}")
for output_path in feedback_logger.iter_output_paths(result):
    print(output_path)

Total feedback records: 3
Failed feedback records: 0
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\19_feedback_log.json
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\19_feedback_log.jsonl
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\19_feedback_report.json
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\19_feedback_audit.csv
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\19_failed_feedback_records.json
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\plots\19_feedback_ratings.png
C:\Users\Prompt\Documents\Hospital Patient Helpdesk Chatbot\hospital_patient_helpdesk_chatbot\01_data\processed\plots\19_fee

## 4. Validate Artifacts

The feedback log should contain one sample record per transcript turn and no invalid records.

In [4]:
records = json.loads(result.feedback_json_path.read_text(encoding="utf-8"))
report = json.loads(result.report_path.read_text(encoding="utf-8"))
failures = json.loads(result.failed_path.read_text(encoding="utf-8"))
jsonl_lines = result.feedback_jsonl_path.read_text(encoding="utf-8").splitlines()

assert len(records) == len(transcript) == 3
assert len(jsonl_lines) == len(records)
assert failures == []
assert report["total_feedback"] == 3
assert report["failed_feedback"] == 0
assert set(report["rating_counts"]).issubset({"thumbs_up", "thumbs_down"})
assert all(record["feedback_id"].startswith("19_FB_") for record in records)

print(json.dumps(report, indent=2))

{
  "generated_at_utc": "2026-06-15T21:04:31.305069+00:00",
  "phase": "19",
  "module": "feedback_logger",
  "module_version": "1.0",
  "transcript_file": "C:\\Users\\Prompt\\Documents\\Hospital Patient Helpdesk Chatbot\\hospital_patient_helpdesk_chatbot\\01_data\\processed\\15_streamlit_transcript.json",
  "total_feedback": 3,
  "failed_feedback": 0,
  "rating_counts": {
    "thumbs_down": 1,
    "thumbs_up": 2
  },
  "reason_tag_counts": {
    "clear": 2,
    "emergency_routing": 1,
    "helpful": 1,
    "missing_detail": 1
  },
  "answer_mode_counts": {
    "emergency": 1,
    "grounded_answer": 2
  },
  "safety_feedback_count": 1,
  "thumbs_up_rate": 0.6667,
  "output_files": [
    "19_feedback_log.json",
    "19_feedback_log.jsonl",
    "19_feedback_report.json",
    "19_feedback_audit.csv",
    "19_failed_feedback_records.json",
    "plots/19_feedback_ratings.png",
    "plots/19_feedback_reason_tags.png"
  ]
}


## 5. Display Feedback Plots

The plots show rating counts and reason-tag counts.

In [5]:
from IPython.display import Image, display

display(Image(filename=str(result.rating_plot_path)))
display(Image(filename=str(result.reason_plot_path)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>


## Notebook and Module Difference

- The notebook is for inspection, validation, and visual review.
- The Python file is the reusable feedback logger and CLI workflow.
- The notebook imports the Python module so both stay aligned.